# Diabetes Scenario 1 — Is HbA1c measured in hospital, and does measuring it go with fewer readmissions?

_Reproducing the core of Strack et al. 2014 on the MIP. Level: first analysis on this dataset. About 90 minutes._

**Before you start.** These are 101,766 inpatient encounters of 71,518 patients with a diabetes
diagnosis in 130 US hospitals, 1999 to 2008, from an electronic health record warehouse: nobody
was randomised, HbA1c was rarely measured, several drug classes of today did not exist, and a
readmission to a hospital outside the network is invisible. One row is **one encounter**; a
patient with several stays appears several times, so the analyses keep the first encounter of
each patient and drop the encounters that ended in death or hospice, as the published analysis
did. Codes: 0 = No, 1 = Yes; Gender 1 = Male, 2 = Female; age only as 10-year bands.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model). No
encounter record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

**Learning goals.** Build the analysis cohort of a published paper with filters; describe a
hospital population; measure how often a test is done; compare an outcome between two groups
with a chi-square; stratify a comparison by a third variable and see an association appear in
one stratum and vanish in another; and explain why an observational association is not a
treatment effect.

**Clinical background.** Glycated haemoglobin (HbA1c) summarises blood glucose over the previous
two to three months. Measuring it when a patient with diabetes is admitted tells the team whether
the patient's usual treatment is working, and is an opportunity to adjust it. In the 2000s this
was rarely done in US hospitals. Strack and colleagues used this dataset to ask two questions:
how often was HbA1c measured, and was measuring it associated with a lower rate of readmission
within 30 days? Their answer, on 69,984 first encounters: measured in 18.4%; readmission 8.7%
when measured versus 9.4% when not; and the association was concentrated in patients admitted
primarily for diabetes. This is an association in observational data: patients in whom HbA1c was
measured may have had more attentive care in general. The paper says so; so should you.

## 1. Connect to the MIP and open the Diabetes pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("Diabetes")    # add version="DiabetesUCI" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
ds = dm.datasets.list()[0]              # one dataset: Diabetes 130-US hospitals 1999-2008
display(dm)
display(ds)

In [ ]:
# The variable tree follows the encounter: Encounter, Admission, Demographics, Diagnoses
# (primary, secondary, summary), Hospital care, Prior utilisation, Laboratory, Diabetes
# medication (individual drugs, summary), Discharge, Outcome.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the datasets of the pathology
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the Diabetes 130 dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ds], variables=variables), filters=filters,
                        handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == POOLED]


def numeric_summary(result, variables):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",)):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    table.attrs["outcome_levels"] = list(lab_y.values())
    table.attrs["events"] = [lab_y.get(e, e) for e in events]
    return table


def collapse(table, groups):
    """Pool the rows of a crosstab into two groups ({name: [row labels]}) and test the 2x2 table
    with a chi-square with Yates' correction, as the platform does, from the aggregate counts."""
    first = table.columns[0]
    levels, events = table.attrs["outcome_levels"], table.attrs["events"]
    rows = []
    for name, members in groups.items():
        part = table[table[first].isin(members)]
        counts = {lev: int(part[lev].sum()) for lev in levels}
        n = sum(counts.values())
        hit = sum(counts[e] for e in events)
        rows.append({first: name, "n": n, **counts, "% " + "/".join(events): 100 * hit / n if n else np.nan})
    out = pd.DataFrame(rows)
    (a, b), (c, d) = [(r["n"] - sum(r[e] for e in events), sum(r[e] for e in events)) for r in rows[:2]]
    n = a + b + c + d
    chi2 = n * (abs(a * d - b * c) - n / 2) ** 2 / ((a + b) * (c + d) * (a + c) * (b + d))
    out.attrs["chi2"], out.attrs["p"] = chi2, math.erfc(math.sqrt(chi2 / 2))
    return out


def band_rates(outcomes, bands, base_filter=None, event="1"):
    """% of encounters with each outcome in bands of a variable: bands maps a label to a filter."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {"band": label}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.1f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(6, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Code | Label in the MIP | Tree location (under Diabetes) | Role | Values |
|---|---|---|---|---|
| `first_encounter` | First encounter of the patient | Encounter | cohort filter: one encounter per patient; curation-derived | 0 No; 1 Yes |
| `death_or_hospice` | Died or discharged to hospice | Discharge | cohort filter: encounters that could be readmitted; curation-derived | 0 No; 1 Yes |
| `A1Cresult` | HbA1c result | Laboratory | exposure: was the test done, and its result | 0 Not measured; 1 Normal (below 7%); 2 Above 7% (7 to 8%); 3 Above 8% |
| `hba1c_group` | HbA1c group (published analysis) | Laboratory | exposure: the four groups of the published analysis; curation-derived | 1 Not measured; 2 Measured, result below 8%; 3 Measured, above 8%, diabetes medication changed; 4 Measured, above 8%, diabetes medication not changed |
| `change` | Change in diabetes medication | Diabetes medication / Summary | outcome of Step 3 | 0 No change; 1 Change in diabetes medication |
| `readmit_30` | Readmitted within 30 days | Outcome | outcome; curation-derived | 0 No; 1 Yes |
| `diag_1_grp` | Primary diagnosis group | Diagnoses / Primary diagnosis | stratification (Step 5); curation-derived | 9 codes, listed in the tree |
| `gender` | Gender | Demographics | description | 1 Male; 2 Female; 9 Unknown/Invalid |
| `race` | Race | Demographics | description | 1 Caucasian; 2 African American; 3 Hispanic; 4 Asian; 5 Other |
| `age_mid` | Age, band midpoint | Demographics | description; curation-derived | integer, 5-95 years |
| `time_in_hospital` | Length of stay | Encounter | description | integer, 1-14 days |
| `n_encounters` | Number of encounters of this patient | Encounter | extension; curation-derived | integer, 1-40 |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `A1Cresult`) is what
the csv column is called; you will see it in some result payloads.

In [ ]:
a1c       = dm.variables["HbA1c result"]
hba1c_grp = dm.variables["HbA1c group (published analysis)"]
change    = dm.variables["Change in diabetes medication"]
readmit   = dm.variables["Readmitted within 30 days"]
diag      = dm.variables["Primary diagnosis group"]
gender    = dm.variables["Gender"]
race      = dm.variables["Race"]
age_mid   = dm.variables["Age, band midpoint"]
stay      = dm.variables["Length of stay"]
display(mip.to_frame([a1c, hba1c_grp, change, readmit, diag, gender, race, age_mid, stay]))

## Step 1. The cohort of the paper, and who is in it

Two filters, kept for every step: the first encounter of each patient (one observation per
patient) and no death or hospice discharge (patients who could be readmitted). One `describe()`
then gives the counts and means.

*Expected:* 69,973 encounters (the paper reports 69,984; the difference is a handful of ties in
the ordering of encounters); 53% women; of the 68,055 with a recorded race, 77% Caucasian and 19%
African American (75% and 18% of all encounters; race is missing for 2.7%); mean age about 65
years (from band midpoints); mean stay 4.3 days, median 3; 9.0% readmitted within 30 days.
Compare with the paper's Table 3: identical to the decimal.

In [ ]:
first = dm.variables["First encounter of the patient"]
death = dm.variables["Died or discharged to hospice"]
cohort = (F(first) == "1") & (F(death) == "0")      # one encounter per patient, able to be readmitted

who = analysis([gender, race, age_mid, stay, readmit], cohort).describe()
display(numeric_summary(who, [age_mid, stay]).style.format({c: "{:.1f}" for c in ("mean", "sd")}).hide(axis="index"))
counts = pd.concat([category_counts(who, v) for v in (gender, race, readmit)], ignore_index=True)
display(counts.style.format({"percent": PCT}).hide(axis="index"))
n_cohort = int(counts.loc[counts["variable"] == readmit.label, "count"].sum())
print(f"{n_cohort:,} encounters in the cohort")

## Step 2. How often was HbA1c measured?

The counts of the four levels of the HbA1c result.

*Expected:* not measured 57,128 (81.6%); measured 12,845 (18.4%): below 7% 3,741, 7 to 8% 2,865,
above 8% 6,239. So 51.4% of measured values were below 8%, as the paper states. One patient in
five with a diabetes diagnosis had the basic diabetes control test during a hospital stay of
several days.

In [ ]:
result = category_counts(analysis([a1c], cohort).describe(), a1c)
display(result.style.format({"percent": PCT}).hide(axis="index"))
measured = result[result["level"] != "Not measured"]
print(f"Measured: {int(measured['count'].sum()):,} ({measured['percent'].sum():.1f}%); "
      f"below 8% among the measured: {100 * measured[measured['level'] != 'Above 8%']['count'].sum() / measured['count'].sum():.1f}%")
ax = result.set_index("level")["percent"].plot(kind="bar", color=["#999999", "#55A868", "#DD8452", "#C44E52"], figsize=(6, 3), rot=0)
ax.set_ylabel("% of encounters"); ax.set_xlabel(""); ax.set_title("HbA1c result, 69,973 first encounters")
plt.tight_layout()
plt.show()

## Step 3. Did the test change management?

The cross-tabulation of the change in diabetes medication by HbA1c result: one `describe()` per
result level for the counts, the *Chi-squared Test* for the p-value.

*Expected:* change in 42.7% when not measured, 43.0% when below 7%, 49.4% when 7 to 8%, and 65.0%
when above 8%; p < 0.001. Paper: 42.5% without test, 55.0% with test, 65.0% when above 8%.
Providers did respond to a high result, but one third of patients with HbA1c above 8% left with
unchanged treatment.

In [ ]:
managed = crosstab(a1c, change, base_filter=cohort, event_label="changed")
show_crosstab(managed, "Diabetes medication changed, by HbA1c result")

## Step 4. Readmission by whether HbA1c was measured

The cross-tabulation of readmission by the paper's four HbA1c groups. The platform tests the
four groups together; the `collapse` helper then pools the three measured groups and tests
"measured versus not measured" on the pooled counts (a chi-square with Yates' correction, as the
platform computes it), since the platform cannot merge categories of a variable by itself.

*Expected:*

| HbA1c group | n | Readmitted within 30 days |
|---|---|---|
| Not measured | 57,128 | 9.1% |
| Below 8% | 6,606 | 8.6% |
| Above 8%, medication changed | 4,058 | 8.6% |
| Above 8%, not changed | 2,181 | 7.4% |

Four-group chi-square p = 0.02; measured overall 8.4% versus not measured 9.1%, p = 0.01. Paper:
8.7% versus 9.4%, p = 0.007. The association is real but small: 7 fewer readmissions per 1,000
encounters.

In [ ]:
by_group = crosstab(hba1c_grp, readmit, base_filter=cohort, event_label="readmitted")
show_crosstab(by_group, "Readmitted within 30 days, by HbA1c group")

groups = labels_of(hba1c_grp)
measured_vs_not = collapse(by_group, {"Not measured": [groups["1"]], "Measured": [groups["2"], groups["3"], groups["4"]]})
display(measured_vs_not.style.format({measured_vs_not.columns[-1]: PCT}).hide(axis="index"))
print(f"Measured versus not measured: chi2 = {measured_vs_not.attrs['chi2']:.2f}, p = {fmt_p(measured_vs_not.attrs['p'])}")

## Step 5. Stratify by the primary diagnosis

The paper's main finding is that the association depends on why the patient was admitted. The
same cross-tabulation three times, with a third filter on the primary diagnosis group: diabetes,
circulatory, respiratory.

*Expected:*

| Primary diagnosis | n | Readmitted, HbA1c not measured | Readmitted, HbA1c measured | p | HbA1c measured |
|---|---|---|---|---|---|
| Diabetes | 5,748 | 10.2% | 7.2% | 0.0002 | 36% |
| Circulatory | 21,384 | 9.5% | 10.2% | 0.21 | 19% |
| Respiratory | 9,486 | 7.7% | 5.5% | 0.001 | 20% |

Among patients admitted for diabetes itself, measuring HbA1c goes with 30 fewer readmissions per
1,000; among patients admitted for heart disease, with none. This is the paper's conclusion,
obtained here with three filters and a cross-tabulation instead of a regression with interaction
terms.

In [ ]:
strata = []
for code_, name in (("4", "Diabetes"), ("1", "Circulatory"), ("2", "Respiratory")):
    t = crosstab(hba1c_grp, readmit, base_filter=cohort & (F(diag) == code_), event_label="readmitted")
    pooled = collapse(t, {"Not measured": [groups["1"]], "Measured": [groups["2"], groups["3"], groups["4"]]})
    pct_col = pooled.columns[-1]
    strata.append({"primary diagnosis": name, "n": int(pooled["n"].sum()),
                   "readmitted, not measured": pooled[pct_col].iloc[0], "readmitted, measured": pooled[pct_col].iloc[1],
                   "p (measured vs not)": pooled.attrs["p"], "p (four groups)": t.attrs["p"],
                   "% measured": 100 * pooled["n"].iloc[1] / pooled["n"].sum()})
strata = pd.DataFrame(strata)
display(strata.style.format({"readmitted, not measured": PCT, "readmitted, measured": PCT, "% measured": PCT,
                             "p (measured vs not)": fmt_p, "p (four groups)": fmt_p}).hide(axis="index"))

ax = strata.set_index("primary diagnosis")[["readmitted, not measured", "readmitted, measured"]].plot(
    kind="bar", figsize=(6, 3.2), color=["#999999", "#4C72B0"], rot=0)
ax.set_ylabel("% readmitted within 30 days"); ax.set_xlabel(""); ax.set_title("Readmission by HbA1c measurement, by primary diagnosis")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Step 6. State the result

A model answer:

> In 69,973 first hospital encounters of patients with a diabetes diagnosis in 130 US hospitals
> (1999 to 2008), HbA1c was measured in only 18%. When it was measured, diabetes medication was
> changed more often (55% versus 43%, 65% when the result was above 8%), and readmission within
> 30 days was slightly less frequent (8.4% versus 9.1%). The association was concentrated in
> patients admitted primarily for diabetes (7.2% versus 10.2%) and absent in patients admitted
> for circulatory disease. Because this is observational data, the finding suggests that
> attention to diabetes during a hospital stay may matter, but it cannot show that ordering the
> test causes fewer readmissions.

**Questions for discussion.** Give two reasons, other than a causal effect of the test, why
patients whose HbA1c was measured might be readmitted less often. In Step 4 the group "above
8%, not changed" has the lowest readmission rate: is this plausible clinically, and what does
the size of that cell (2,181) do to the precision of the percentage? Why did the authors keep
only the first encounter per patient?

**Optional extension.** Repeat Step 4 on all encounters that could be readmitted: the
readmission rate rises to about 11% because frequent patients appear many times. Then look at
readmission by the number of encounters of the patient, in bands.

In [ ]:
could_be_readmitted = F(death) == "0"
all_enc = crosstab(hba1c_grp, readmit, base_filter=could_be_readmitted, event_label="readmitted")
display(all_enc.style.format({"% readmitted": PCT}).hide(axis="index"))
print(f"All {int(all_enc['n'].sum()):,} encounters: readmitted within 30 days "
      f"{100 * all_enc['Yes'].sum() / all_enc['n'].sum():.1f}%")

n_enc = dm.variables["Number of encounters of this patient"]
bands = {"1 encounter": F(n_enc) == 1, "2": F(n_enc) == 2, "3 to 5": F(n_enc).between(3, 5), "6 or more": F(n_enc) >= 6}
display(band_rates([readmit], bands, base_filter=could_be_readmitted).style.format({readmit.label: PCT}).hide(axis="index"))